# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

In [1]:
!pip install kagglehub pandas scikit-learn rank_bm25 nltk


   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protob


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\fredd\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [2]:
import kagglehub

# Descargar dataset
path = kagglehub.dataset_download(
    "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects"
)

print("Path to dataset files:", path)

100%|██████████| 18.7M/18.7M [00:40<00:00, 485kB/s]

Extracting files...


Path to dataset files: C:\Users\fredd\.cache\kagglehub\datasets\gzdekzlkaya\wikipedia-text-corpus-for-nlp-and-llm-projects\versions\1


In [ ]:
# Ver archivos en el directorio del dataset
import os

print(os.listdir(path))

['wikipedia_text_corpus.csv']


In [4]:
import pandas as pd

# Cargar dataset
df = pd.read_csv(f"{path}/wikipedia_text_corpus.csv")

# Ver primeras filas
df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


In [ ]:
# Ver columnas del DataFrame
print(df.columns)

Index(['Unnamed: 0', 'text'], dtype='object')


In [10]:
# Ver número de filas y columnas
print(df.shape)

(10859, 2)


In [ ]:
# Eliminar columna 'Unnamed: 0' 
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
# Carga los Stop Words
import re
import nltk

from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fredd\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Crear conjunto de stop words
stop_words = set(stopwords.words('english'))

def preprocess(text):

    text = str(text).lower()

    # eliminar caracteres especiales
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # tokenizar
    tokens = text.split()

    # eliminar stopwords
    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

In [ ]:
# Se crea una nueva columna 'clean_text' aplicando la función de preprocesamiento a la columna 'text'
df['clean_text'] = df['text'].apply(preprocess)

In [ ]:
print(df.columns)

Index(['text', 'clean_text'], dtype='object')


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [ ]:
# Importa la clase TfidfVectorizer para convertir el texto en vectores numéricos
from sklearn.feature_extraction.text import TfidfVectorizer
# Importa la función cosine_similarity para calcular la similitud entre vectores
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Se crea un vectorizador TF-IDF
vectorizer = TfidfVectorizer()
# Convertir la columna 'clean_text' en una matriz TF-IDF
tfidf_matrix = vectorizer.fit_transform(df['clean_text'])
# Mostrar la dimensión de la matriz TF-IDF
print("Dimensión matriz TF-IDF:")
print(tfidf_matrix.shape)

# Ejemplo de las consultas que se van a realizar queries
queries = [
    "machine learning",
    "artificial intelligence",
    "computer science",
    "world war",
    "space exploration",
    "climate change",
    "renewable energy",
    "quantum physics",
    "human anatomy",
    "history of europe"
]
# Función para buscar documentos relevantes usando TF-IDF y similitud coseno
def search_tfidf(query, top_k=5):

    # convertir query en vector
    query_vector = vectorizer.transform([query])

    # calcular similitud coseno
    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # obtener índices de documentos más relevantes
    top_indices = similarities.argsort()[-top_k:][::-1]

    # devolver documentos
    return df.iloc[top_indices]
# Realizar consultas y mostrar resultados
for query in queries:    
    print("\n" + "="*60)    
    print("QUERY:", query)    
    print("="*60)    
    # Buscar documentos relevantes usando TF-IDF
    results = search_tfidf(query)    
    # Mostrar resultados
    for rank, (i, row) in enumerate(results.iterrows(), start=1):        
        print(f"\nTop {rank} Documento Recuperado:\n")        
        # mostrar primeros 300 caracteres        
        print(row['text'][:300])        
        print("\n" + "-"*50)

Dimensión matriz TF-IDF:
(10859, 173939)

QUERY: machine learning

Top 1 Documento Recuperado:

Outline of machine learning

The following outline is provided as an overview of and topical guide to machine learning. Machine learning is a subfield of soft computing within computer science that evolved from the study of pattern recognition and computational learning theory in artificial intellig

--------------------------------------------------

Top 2 Documento Recuperado:

Digital learning

Digital learning is any type of learning that is accompanied by technology or by instructional practice that makes effective use of technology. It encompasses the application of a wide spectrum of practices including: blended and virtual learning. 

Sometimes confused with online l

--------------------------------------------------

Top 3 Documento Recuperado:

Virtual learning environment

A virtual learning environment (VLE) in educational technology is a Web-based platform for the digital aspec

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [24]:
!pip install rank_bm25


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\fredd\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# Importa la clase BM25Okapi para implementar el modelo BM25
from rank_bm25 import BM25Okapi

In [ ]:
# Tokenizar los documentos para BM25 
tokenized_corpus = [
    doc.split()
    for doc in df['clean_text']
]

In [ ]:
# Crear modelo BM25
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
# Función para buscar documentos relevantes usando BM25
def search_bm25(query, top_k=5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = scores.argsort()[-top_k:][::-1]
    return df.iloc[top_indices]

In [ ]:
# Se realizan las mismas consultas  las 10 usando BM25 y se muestran los resultados
for query in queries:

    print("\n" + "="*60)
    print("QUERY:", query)
    print("="*60)
    results = search_bm25(query)
    for rank, (i, row) in enumerate(results.iterrows(), start=1):
        print(f"\nTop {rank} Documento Recuperado:\n")
        print(row['text'][:300])
        print("\n" + "-"*50)


QUERY: machine learning

Top 1 Documento Recuperado:

Outline of machine learning

The following outline is provided as an overview of and topical guide to machine learning. Machine learning is a subfield of soft computing within computer science that evolved from the study of pattern recognition and computational learning theory in artificial intellig

--------------------------------------------------

Top 2 Documento Recuperado:

Teaching machine

Teaching machines were originally mechanical devices. They presented educational materials and taught students. They were first invented by Sidney L. Pressey in the mid-1920s. His machine originally administered multiple-choice questions. The machine could be set so it moved on onl

--------------------------------------------------

Top 3 Documento Recuperado:

Ryszard S. Michalski

Ryszard S. Michalski (May 7, 1937 â€“ September 20, 2007) was a Polish-American computer scientist. Michalski was Professor at George Mason University and a 

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [ ]:
# Consulta que se utilizará para comparar ambos modelos
query = "machine learning"

# Mostrar los resultados obtenidos con TF-IDF 
print("="*60)
print("RESULTADOS TF-IDF")
print("="*60)

tfidf_results = search_tfidf(query)

for rank, (i, row) in enumerate(tfidf_results.iterrows(), start=1):
    print(f"\nTop {rank} Documento:\n")
    print(row['text'][:300])
    print("\n" + "-"*50)

# Se una print("\n\n") para separar los resultados de ambos modelos
print("\n\n")

# Mostrar resultados obtenidos con BM25 para la consulta "machine learning"
# Se usa el "="*60 para crear una línea de separación visual
print("="*60)
print("RESULTADOS BM25")
print("="*60)

bm25_results = search_bm25(query)
for rank, (i, row) in enumerate(bm25_results.iterrows(), start=1):
    print(f"\nTop {rank} Documento:\n")
    # Mostrar una parte del documento
    print(row['text'][:300])
    print("\n" + "-"*50)

RESULTADOS TF-IDF

Top 1 Documento:

Outline of machine learning

The following outline is provided as an overview of and topical guide to machine learning. Machine learning is a subfield of soft computing within computer science that evolved from the study of pattern recognition and computational learning theory in artificial intellig

--------------------------------------------------

Top 2 Documento:

Digital learning

Digital learning is any type of learning that is accompanied by technology or by instructional practice that makes effective use of technology. It encompasses the application of a wide spectrum of practices including: blended and virtual learning. 

Sometimes confused with online l

--------------------------------------------------

Top 3 Documento:

Virtual learning environment

A virtual learning environment (VLE) in educational technology is a Web-based platform for the digital aspects of courses of study, usually within educational institutions. They present res

### CONCLUSION DE AMBOS METODOS USADOS. -

En la comparación de resultados se observó que ambos modelos recuperaron documentos relacionados con la consulta “machine learning”. Sin embargo, el orden y la relevancia de los documentos fueron diferentes.

TF-IDF recuperó documentos relacionados con el término “learning”, mostrando resultados más generales sobre aprendizaje digital y entornos virtuales de aprendizaje. Esto ocurre porque TF-IDF se basa principalmente en la frecuencia e importancia relativa de las palabras dentro de los documentos.

Por otro lado, BM25 recuperó documentos más específicos relacionados directamente con “machine learning”, incluyendo autores y conceptos asociados al área. Esto se debe a que BM25 considera mejor la frecuencia de los términos y la longitud de los documentos, obteniendo resultados más precisos en las primeras posiciones.

En conclusión, ambos modelos permiten recuperar información relevante, pero BM25 presentó una recuperación más precisa para consultas específicas.